In [2]:
import Shadow
import logging
import numpy as np
from random import random
from ophyd import Signal, Component, Device
from ophyd.signal import InternalSignal
from optlnls.plot import plot_beam
from optlnls.importing import read_shadow_beam
from bluesky import RunEngine
from bluesky.plans import scan
from bluesky.callbacks.best_effort import BestEffortCallback
from bluesky.callbacks.tiled_writer import TiledWriter
from tiled.client import simple
from blop.plans import optimize
from blop.protocols import EvaluationFunction, Optimizer, OptimizationProblem

RE = RunEngine()

tiled_client = simple()
tiled_writer = TiledWriter(tiled_client)
RE.subscribe(tiled_writer)

bec = BestEffortCallback()
bec.disable_plots()
RE.subscribe(bec)

logging.getLogger("httpx").setLevel(logging.WARNING)

[WARNING 04-13 09:57:36] ax.service.utils.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.
2026-04-13 09:57:36.951 INFO: setup plugin alembic.autogenerate.schemas
2026-04-13 09:57:36.953 INFO: setup plugin alembic.autogenerate.tables
2026-04-13 09:57:36.953 INFO: setup plugin alembic.autogenerate.types
2026-04-13 09:57:36.954 INFO: setup plugin alembic.autogenerate.constraints
2026-04-13 09:57:36.955 INFO: setup plugin alembic.autogenerate.defaults
2026-04-13 09:57:36.955 INFO: setup plugin alembic.autogenerate.comments
2026-04-13 09:57:40.697 INFO: Subprocess stdout: 
2026-04-13 09:57:40.698 INFO: Subprocess stderr: Database sqlite+aiosqlite:////tmp/tmpvx6rrx2z/catalog.db is new. Creating tables.
Database initialized.

Tiled version 0.2.8
2026-04-13 09:57:41.659 INFO: 

http://127.0.0.1:44667/api/v1?api_key=2ba8f9b45babeeb5


2026-04-13 09:57:41.948 INFO: HTTP Request: GET http://127.0.0.1:44667/api/v1/metadata/?include_data_sources=true "HTTP/1.1 200 OK"


In [3]:
class ShadowEnergy(Signal):

    def set(self, value: float, **kwargs):
        sts = super().set(value, **kwargs)
        self.parent.oe.PH1 = value - self.parent.delta_energy / 2
        self.parent.oe.PH2 = value + self.parent.delta_energy / 2
        return sts

class ShadowAttribute(Signal):

    def set(self, value: float, **kwargs):
        sts = super().set(value, **kwargs)
        oe = self.parent.set_standard_parameters(self.parent.energy.get())
        self.parent.oe = oe
        return sts


class ShadowSource(Device):

    energy = Component(ShadowEnergy, kind="hinted")
    sigmax = Component(ShadowAttribute, kind="hinted")

    def __init__(
        self, name: str, energy: float, delta_energy: float, n_rays: int = 1_000_000
    ):
        super().__init__(name=name)
        self.delta_energy = delta_energy
        self.n_rays = n_rays
        self.oe = self.set_standard_parameters(energy)
        self.energy.set(energy).wait()
        self.sigmax.set(0.033).wait()

    def set_standard_parameters(self, energy):
        oe = Shadow.Source()
        oe.SIGDIX = 0.01
        oe.SIGDIZ = 0.00113
        oe.SIGMAZ = 0.027
        oe.SIGMAX = self.sigmax.get()
        oe.VDIV1 = 0.001
        oe.VDIV2 = 0.001
        oe.FDISTR = 3
        oe.F_COLOR = 3
        oe.F_PHOT = 0
        oe.HDIV1 = 0.004
        oe.HDIV2 = 0.004
        oe.IDO_VX = 0
        oe.IDO_VZ = 0
        oe.IDO_X_S = 0
        oe.IDO_Y_S = 0
        oe.IDO_Z_S = 0
        oe.ISTAR1 = 5676561
        oe.NPOINT = self.n_rays
        oe.PH1 = energy - self.delta_energy / 2
        oe.PH2 = energy + self.delta_energy / 2
        return oe

    def read(self, **kwargs):
        self.beam = Shadow.Beam()
        self.beam.genSource(self.oe)
        # beam2D = read_shadow_beam(
        #         beam=self.beam,
        #         x_column_index=3,
        #         y_column_index=1
        # )
        # plot_beam(
        #     beam2D=beam2D,
        #     show_plot=True,
        #     fitType=3,
        #     textA=2,textB=5,
        #     x_range=1,x_range_min=-0.1,x_range_max=0.1,
        #     y_range=1,y_range_min=-0.1,y_range_max=0.1,
        #     zero_pad_x=2
        # )
        return super().read(**kwargs)


In [4]:
class ShadowM1(Device):
    def __init__(self, past_element: list, **kwargs):
        super().__init__(**kwargs)
        self.past_element = past_element
        self.idx = 0
        self.generate_oe()

    def generate_oe(self):
        oe = Shadow.OE()
        oe.ALPHA = 90.0
        oe.CIL_ANG = 90.0
        oe.DUMMY = 0.1
        oe.FCYL = 1
        oe.FHIT_C = 1
        oe.FMIRR = 1
        oe.FWRITE = 1
        oe.F_EXT = 1
        oe.RLEN1 = 180.0
        oe.RLEN2 = 180.0
        oe.RMIRR = 2108.8753452152337
        oe.RWIDX1 = 20.0
        oe.RWIDX2 = 20.0
        oe.T_IMAGE = 35.0
        oe.T_INCIDENCE = 80.825
        oe.T_REFLECTION = 80.825
        oe.T_SOURCE = 0.0
        self.oe = oe

    def read(self, **kwargs):
        self.generate_oe()
        self.past_element.read()
        self.beam = self.past_element.beam
        self.beam.traceOE(self.oe, self.idx)
        # beam2D = read_shadow_beam(
        #         beam=self.beam,
        #         x_column_index=3,
        #         y_column_index=1
        # )
        # plot_beam(
        #     beam2D=beam2D,
        #     show_plot=True,
        #     fitType=3,
        #     textA=2,textB=5,
        #     x_range=1,x_range_min=-0.2,x_range_max=0.2,
        #     y_range=1,y_range_min=-0.2,y_range_max=0.2,
        #     zero_pad_x=2
        # )
        return super().read(**kwargs)

In [5]:
class ShadowPitch(Signal):
    def set(self, value: float, **kwargs):
        sts = super().set(value, **kwargs)
        self.parent.generate_oe(value)
        return sts

class ShadowM2(Device):
    pitch = Component(ShadowPitch, kind="hinted")
    center_x = Component(InternalSignal, kind="hinted")

    def __init__(self, past_element: list, pitch: float, **kwargs):
        super().__init__(**kwargs)
        self.idx = 1
        self.past_element = past_element
        self.generate_oe(pitch)

    def generate_oe(self, pitch: float):
        oe = Shadow.OE()
        oe.ALPHA = 180.0
        oe.DUMMY = 0.1
        oe.FCYL = 1
        oe.FHIT_C = 1
        oe.FMIRR = 1
        oe.FWRITE = 1
        oe.F_DEFAULT = 0
        oe.F_MOVE = 1
        oe.RLEN1 = 300.0
        oe.RLEN2 = 300.0
        oe.RWIDX1 = 20.0
        oe.RWIDX2 = 20.0
        oe.SIMAG = 14095.0
        oe.SSOUR = 14095.0
        oe.THETA = 82.75
        oe.T_IMAGE = 19.05
        oe.T_INCIDENCE = 82.75
        oe.T_REFLECTION = 82.75
        oe.T_SOURCE = 0.0
        oe.X_ROT = np.degrees(pitch)
        self.oe = oe

    def describe(self, **kwargs):
        new_describe = super().describe()
        new_describe[self.center_x.name] ={'source': 'Center X',
                                'dtype': 'number',
                                'shape': [],
                                'units': '',
                                'lower_ctrl_limit': 0.0,
                                'upper_ctrl_limit': 0.0,
                                'precision': 3}
        return new_describe
    
    def read(self, **kwargs):
        self.generate_oe(self.pitch.get())
        self.past_element.read()
        self.beam = self.past_element.beam
        self.beam.traceOE(self.oe, self.idx)
        # beam2D = read_shadow_beam(
        #         beam=self.beam,
        #         x_column_index=3,
        #         y_column_index=1
        # )
        # plot_beam(
        #     beam2D=beam2D,
        #     show_plot=True,
        #     fitType=3,
        #     textA=2,textB=5,
        #     x_range=1,x_range_min=-0.4,x_range_max=0.3,
        #     y_range=1,y_range_min=-0.4,y_range_max=0.3,
        #     zero_pad_x=2
        # )
        beam_info = self.beam.histo2(col_h = 3, col_v = 1, nbins_h = 100, nbins_v = 100, nolost = 1, ref = 23)
        self.center_x.put((beam_info['xrange'][1] + beam_info['xrange'][0])/2, internal=True)
        return super().read(**kwargs)

In [6]:
source = ShadowSource(
    name="src",
    energy=20000,
    delta_energy=1000,
    n_rays=100_000
)

m1 = ShadowM1(
    name="m1",
    past_element=source
)

m2 = ShadowM2(
        name="m2",
        pitch=0.001,
        past_element=m1
)

In [7]:
class DetectorEvaluation(EvaluationFunction):
    def __init__(self, tiled_client):
        self.tiled_client = tiled_client
    
    def __call__(self, uid: str, suggestions: list[dict]) -> list[dict]:
        outcomes = []
        run = self.tiled_client[uid]
        pitch = run["primary/internal/m2_pitch"].read()[0]
        center_x = run["primary/internal/m2_center_x"].read()
        suggestion_ids = run.metadata["start"]["blop_suggestion_ids"]
        
        for idx, sid in enumerate(suggestion_ids):
            outcome = {
                "_id": sid,
                "center_x": center_x,
                # 'intensity': intensity['intensity'],
                # 'fwhm_h' : intensity['fwhm_h'],
                # 'fwhm_v' : intensity['fwhm_v'],
                # 'x_center' : intensity['xrange'][1] - intensity['xrange'][0],
                # 'y_center' : intensity['yrange'][1] - intensity['yrange'][0],
                'pitch' : pitch
            }
            outcomes.append(outcome)
        return outcomes

In [8]:
class M2Optimizer(Optimizer):

    outcome = set()
    dof = set()
    idx = 0

    def __init__(self):
        self.pitch_dof = [-0.01, 0.01]
        self.step = 0.001

    def convert_to_dof(self, value: float):
        nrange = abs(self.pitch_dof[1] - self.pitch_dof[0])
        return (value * nrange) + self.pitch_dof[0]

    def suggest(self, num_points: int | None = None) -> list[dict]:
        self.idx += num_points
        if len(self.outcome) == 0:
            pitch = self.convert_to_dof(random())
        else:
            if self.outcome[0]["center_x"] > 0:
                pitch = self.outcome[0]["pitch"] + self.step
            else:
                pitch = self.outcome[0]["pitch"] - self.step
        return [{
            "_id": self.idx,
            "m2_pitch": pitch
        }]

    def ingest(self, points: list[dict]) -> None:
        self.outcome = points

In [16]:
optimization_problem = OptimizationProblem(
    readables=[m2],
    movables=[m2.pitch],
    optimizer=M2Optimizer(),
    evaluation_function=DetectorEvaluation(tiled_client)
)

In [17]:
RE(optimize(optimization_problem, iterations=5, n_points=1))

2026-04-13 10:02:27.060 INFO: Executing plan <bluesky.utils.Plan object at 0x71eeb10fd4e0>
2026-04-13 10:02:27.063 INFO: Change state on <bluesky.run_engine.RunEngine object at 0x71eebc5e4b50> from 'idle' -> 'running'




Transient Scan ID: 41     Time: 2026-04-13 10:02:27
Persistent Unique Scan ID: '09c673e7-0a51-4822-b7cb-20fb1e9ae857'
 Generated         5000  rays out of       100000
                  10000
                  15000
                  20000
                  25000
                  30000
                  35000
                  40000
                  45000
                  50000
                  55000
                  60000
                  65000
                  70000
                  75000
                  80000
                  85000
                  90000
                  95000
                 100000
 Exit from SOURCE
 Call to RESET
 Exit from RESET
 Call to SETSOUR
 Exit from SETSOUR
 Call to IMREF
 Exit from IMREF
 Call to OPTAXIS
 Exit from OPTAXIS
 Call to MSETUP
 Exit from MSETUP
 Call to RESTART
 Exit from RESTART
 Call to MIRROR
 Exit from MIRROR
 Call to IMAGE
 Exit from IMAGE
 Call to DEALLOC
 Exit from DEALLOC
 Call to RESET
 Exit from RESET
 Call to SETSOUR

2026-04-13 10:02:33.301 INFO: Change state on <bluesky.run_engine.RunEngine object at 0x71eebc5e4b50> from 'running' -> 'idle'
2026-04-13 10:02:33.301 INFO: Cleaned up from plan <bluesky.utils.Plan object at 0x71eeb10fd4e0>


('09c673e7-0a51-4822-b7cb-20fb1e9ae857',
 '2d6ddc69-9181-40af-9428-595811d13f76',
 '2fd5dc9c-636a-45c7-9e73-a24a9a7adb00',
 '6fbd5024-fee6-4a9e-9538-ef5d39b33c50',
 '5f3f042d-18f0-4756-aaf8-1826727fc881')